<a href="https://colab.research.google.com/github/ryanaxiom/Applied-ML/blob/main/code/Day21_Carnegie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Instructions**: Before you begin, click **File > Save a copy in Drive** so you do not lose your work!

In [2]:
url = ("https://raw.githubusercontent.com/ryanaxiom/Applied-ML/main/data/carnegie_data.csv")

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
carnegie_raw = pd.read_csv(url, encoding="cp1252")
carnegie = carnegie_raw.copy()
carnegie["research"] = carnegie["serd"] + carnegie["nonserd"]
d = carnegie[carnegie["research"].notna()].copy()
kf = KFold(n_splits=5, shuffle=True, random_state=309)
cols = ["fallenr20", "grtenr20", "docrsdeg", "totdeg", "rooms"]
print(carnegie.shape, d.shape)

(3939, 102) (302, 102)


In [3]:
nh_url = url.replace("carnegie_data.csv", "nhanes_adults.csv")
nh = pd.read_csv(nh_url)
yd = (nh["Diabetes"] == "Yes").astype(int)   # told by a doctor
nh["male"] = (nh["Sex"] == "male").astype(int)
exam = ["Age", "BMI", "BPSysAve", "BPDiaAve", "Pulse",
        "TotChol", "DirectChol", "male"]
screen = make_pipeline(SimpleImputer(strategy="median"),
                       StandardScaler(), LogisticRegression())
p_nh = cross_val_predict(screen, nh[exam], yd, cv=kf,
                         method="predict_proba")[:, 1]
auc = cross_val_score(screen, nh[exam], yd, cv=kf,
                      scoring="roc_auc")
print(nh.shape, int(yd.sum()), round(auc.mean(), 4))

(11769, 22) 1668 0.7942


In [4]:
rule = (nh["Age"].between(35, 70) & (nh["BMI"] >= 25)).astype(int)
k = int(rule.sum())                         # tests the rule orders
top = (p_nh >= np.sort(p_nh)[-k]).astype(int)   # model's top k
for name, call in [("USPSTF rule", rule), ("model, top k", top)]:
    tn, fp, fn, tp = confusion_matrix(yd, call).ravel()
    print(name, int(call.sum()), tp, round(tp / (tp + fn), 3),
          round(fp / (fp + tn), 3))

USPSTF rule 4986 962 0.577 0.398
model, top k 4986 1347 0.808 0.36


In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text
tree = make_pipeline(SimpleImputer(strategy="median"),
                     DecisionTreeClassifier(max_depth=2,
                                            random_state=309))
tree.fit(nh[exam], yd)
print(export_text(tree[-1], feature_names=exam, show_weights=True))
auc_t = cross_val_score(tree, nh[exam], yd, cv=kf,
                        scoring="roc_auc")
print(round(auc_t.mean(), 4))

|--- Age <= 49.50
|   |--- Age <= 40.50
|   |   |--- weights: [4156.00, 113.00] class: 0
|   |--- Age >  40.50
|   |   |--- weights: [1588.00, 184.00] class: 0
|--- Age >  49.50
|   |--- TotChol <= 4.33
|   |   |--- weights: [758.00, 543.00] class: 0
|   |--- TotChol >  4.33
|   |   |--- weights: [3599.00, 828.00] class: 0

0.7454


In [6]:
filled = nh[exam].fillna(nh[exam].median())
share = filled.var() / filled.var().sum()   # of squared distance
print(share.round(3).to_dict())
for name, steps in [("raw", []), ("scaled", [StandardScaler()])]:
    knn = make_pipeline(SimpleImputer(strategy="median"), *steps,
                        KNeighborsClassifier(n_neighbors=101))
    auc = cross_val_score(knn, nh[exam], yd, cv=kf,
                          scoring="roc_auc")
    ll = cross_val_score(knn, nh[exam], yd, cv=kf,
                         scoring="neg_log_loss")
    print(name, round(auc.mean(), 4), round(-ll.mean(), 4))

{'Age': 0.325, 'BMI': 0.046, 'BPSysAve': 0.321, 'BPDiaAve': 0.168, 'Pulse': 0.138, 'TotChol': 0.001, 'DirectChol': 0.0, 'male': 0.0}
raw 0.7828 0.3716
scaled 0.8023 0.3443


In [7]:
for k in [301, 101, 25, 1]:
    knn = make_pipeline(SimpleImputer(strategy="median"),
                        StandardScaler(),
                        KNeighborsClassifier(n_neighbors=k))
    auc = cross_val_score(knn, nh[exam], yd, cv=kf,
                          scoring="roc_auc")
    ll = cross_val_score(knn, nh[exam], yd, cv=kf,
                         scoring="neg_log_loss")
    print(k, round(auc.mean(), 4), round(-ll.mean(), 4))
knn.fit(nh[exam], yd)                # knn is the k = 1 pipeline
print(round((knn.predict(nh[exam]) == yd).mean(), 4))  # in-sample

301 0.8027 0.3371
101 0.8023 0.3443
25 0.7844 0.4577
1 0.5858 7.1358
0.9962


In [8]:
from sklearn.metrics import log_loss
naive = log_loss(yd, np.full(len(yd), yd.mean()))
print(round(naive, 4), round(7.1358 / naive, 1))
everyone = make_pipeline(SimpleImputer(strategy="median"),
                         StandardScaler(),
                         KNeighborsClassifier(n_neighbors=9415))
ll_all = cross_val_score(everyone, nh[exam], yd, cv=kf,
                         scoring="neg_log_loss")
print(round(-ll_all.mean(), 4))

0.4081 17.5
0.4083


In [9]:
near = make_pipeline(SimpleImputer(strategy="median"),
                     StandardScaler(),
                     KNeighborsClassifier(n_neighbors=301))
for name, model in [("logistic", screen), ("neighbors", near)]:
    auc = cross_val_score(model, nh[exam], yd, cv=kf,
                          scoring="roc_auc")
    ll = cross_val_score(model, nh[exam], yd, cv=kf,
                         scoring="neg_log_loss")
    print(name, round(auc.mean(), 4), round(-ll.mean(), 4))

logistic 0.7942 0.3391
neighbors 0.8027 0.3371


In [10]:
print(round(0.8027 - 0.7942, 4), round(0.3391 - 0.3371, 4))
screen.fit(nh[exam], yd)
print(screen[-1].coef_.size + screen[-1].intercept_.size,
      [len(train) for train, test in kf.split(nh)])

0.0085 0.002
9 [9415, 9415, 9415, 9415, 9416]


In [11]:
names = {1: "public", 2: "nonprofit", 3: "for-profit"}  # codebook
ctl = carnegie["control"].map(names)
print(ctl.value_counts().to_dict(),
      round(ctl.value_counts(normalize=True).max(), 4))

{'nonprofit': 1666, 'public': 1628, 'for-profit': 645} 0.4229


In [12]:
from sklearn.metrics import log_loss
naive3 = []
for train, test in kf.split(carnegie):
    shares = ctl.iloc[train].value_counts(normalize=True).sort_index()
    naive3.append(log_loss(ctl.iloc[test],
                           np.tile(shares.values, (len(test), 1)),
                           labels=list(shares.index)))
print(round(np.mean(naive3), 4))

1.0262


In [13]:
three = make_pipeline(FunctionTransformer(np.log1p),
                      StandardScaler(), LogisticRegression())
acc = cross_val_score(three, carnegie[cols], ctl, cv=kf,
                      scoring="accuracy")
pred = cross_val_predict(three, carnegie[cols], ctl, cv=kf)
labels = ["public", "nonprofit", "for-profit"]
cm = confusion_matrix(ctl, pred, labels=labels)  # rows = truth
print(round(acc.mean(), 4))
print(cm)
print((cm.diagonal() / cm.sum(axis=1)).round(3))  # recall by row

0.7687
[[1313  248   67]
 [ 193 1332  141]
 [ 113  149  383]]
[0.807 0.8   0.594]


In [14]:
near3 = make_pipeline(FunctionTransformer(np.log1p),
                      StandardScaler(),
                      KNeighborsClassifier(n_neighbors=25))
for name, model in [("logistic", three), ("neighbors", near3)]:
    out = []
    for metric in ["accuracy", "roc_auc_ovr", "neg_log_loss"]:
        s = cross_val_score(model, carnegie[cols], ctl, cv=kf,
                            scoring=metric)
        out.append(round(float(abs(s.mean())), 4))
    print(name, out)

logistic [0.7687, 0.904, 0.565]
neighbors [0.7878, 0.9148, 0.8501]
